# Foundation Models & Scaling Laws

**Companion lesson:** https://ml-viz.vercel.app/courses/transformers/05-foundation-models-and-scaling

A runnable tour of the Chinchilla loss law $L(N, D) = A/N^\alpha + B/D^\beta + E$:
we fit it from synthetic (N, D, L) measurements, derive the compute-optimal
allocation $D \approx 20 N$ from the fit, and plot loss-vs-compute curves on a
log-log axis with the compute-optimal points marked. Pure NumPy + matplotlib.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor']   = '#1a1d27'
plt.rcParams['text.color']       = '#e2e8f0'
plt.rcParams['axes.labelcolor']  = '#e2e8f0'
plt.rcParams['xtick.color']      = '#94a3b8'
plt.rcParams['ytick.color']      = '#94a3b8'
plt.rcParams['axes.edgecolor']   = '#334155'
plt.rcParams['axes.grid']        = True
plt.rcParams['grid.color']       = '#1e293b'
plt.rcParams['figure.figsize']   = (9, 5)
np.random.seed(0)

## 1 — The Chinchilla loss law

The Hoffmann et al. (2022) joint fit:

$$L(N, D) = \frac{A}{N^{\alpha}} + \frac{B}{D^{\beta}} + E$$

with $N$ = parameter count, $D$ = training tokens, and rough exponents
$\alpha \approx 0.34$, $\beta \approx 0.28$. We start by evaluating it on a
grid and plotting some curves to build intuition.

In [ ]:
# Ground-truth constants we will pretend are the unknowns we are fitting.
A_true, B_true, E_true = 406.4, 410.7, 1.69
alpha_true, beta_true  = 0.34, 0.28

def chinchilla_loss(N, D, A=A_true, B=B_true, E=E_true,
                    alpha=alpha_true, beta=beta_true):
    return A / N**alpha + B / D**beta + E

# Quick sanity check at a handful of (N, D) combos.
for N, D in [(1e9, 2e10), (7e9, 1.4e11), (7e10, 1.4e12)]:
    print(f"N={N:.0e}, D={D:.0e}  ->  L = {chinchilla_loss(N, D):.3f}")

## 2 — Fitting the law from synthetic measurements

Imagine we have run a sweep of (N, D) training jobs and recorded the final loss.
We will treat $A$, $B$, $E$ as unknowns at fixed exponents (a common practical
simplification) and recover them via plain NumPy least squares.

Writing $u = N^{-\alpha}$ and $v = D^{-\beta}$, the model is **linear** in
$(A, B, E)$:

$$L = A \cdot u + B \cdot v + E \cdot 1.$$

So one `np.linalg.lstsq` recovers all three.

In [ ]:
# Build a noisy training-sweep dataset.
Ns = np.array([1e8, 3e8, 1e9, 3e9, 1e10, 3e10, 1e11])
Ds = np.array([1e9, 5e9, 2e10, 1e11, 5e11, 2e12])

rows = []
for N in Ns:
    for D in Ds:
        L = chinchilla_loss(N, D) * np.exp(np.random.randn() * 0.01)  # 1% noise
        rows.append((N, D, L))
rows = np.array(rows)
N_obs, D_obs, L_obs = rows[:, 0], rows[:, 1], rows[:, 2]

# Solve  A * N^-alpha + B * D^-beta + E = L   in least-squares sense.
X = np.column_stack([
    N_obs ** (-alpha_true),
    D_obs ** (-beta_true),
    np.ones_like(L_obs),
])
coef, *_ = np.linalg.lstsq(X, L_obs, rcond=None)
A_fit, B_fit, E_fit = coef

print(f"true: A={A_true:.2f}  B={B_true:.2f}  E={E_true:.3f}")
print(f"fit : A={A_fit:.2f}  B={B_fit:.2f}  E={E_fit:.3f}")

# Sanity: fitted predictions vs observed loss
pred = X @ coef
resid_pct = 100.0 * np.abs(pred - L_obs) / L_obs
print(f"mean abs residual: {resid_pct.mean():.2f}%  max: {resid_pct.max():.2f}%")

## 3 — Deriving the compute-optimal allocation

Total training compute (forward + backward + activations) for a dense
Transformer is well approximated by

$$C \approx 6 \cdot N \cdot D \text{ FLOPs.}$$

Under the Chinchilla rule $D = 20 N$, this becomes

$$C = 6 \cdot N \cdot 20 N = 120 N^2 \;\Rightarrow\; N^* = \sqrt{C / 120}, \quad D^* = 20 N^*.$$

We will use this closed form directly. (A full derivation would set
$\partial L / \partial N = \partial L / \partial D \cdot 20$ along the constraint
$D = 20 N$ and solve; the 20 comes from the fitted exponents.)

In [ ]:
def chinchilla_optimal_demo(C):
    """Quick demo using the closed form  N* = sqrt(C / 120),  D* = 20 N*."""
    N_opt = np.sqrt(C / 120.0)
    D_opt = 20.0 * N_opt
    return N_opt, D_opt

for C in [1e21, 1e22, 1e23, 1e24]:
    N_opt, D_opt = chinchilla_optimal_demo(C)
    L_opt = chinchilla_loss(N_opt, D_opt, A=A_fit, B=B_fit, E=E_fit)
    print(f"C = {C:.0e} FLOPs  ->  N* = {N_opt/1e9:6.2f}B params,  "
          f"D* = {D_opt/1e9:7.0f}B tokens,  predicted L = {L_opt:.2f}")

## 4 — Loss vs compute (log-log) with optimal points marked

For three fixed model sizes (1B, 7B, 70B parameters) we sweep $D$ from a small
token budget up to a very large one, plot $L$ vs compute $C = 6 N D$, and mark
the Chinchilla-optimal point per curve.

In [ ]:
model_sizes = [(1e9, '1B', '#14b8a6'),
               (7e9, '7B', '#6366f1'),
               (7e10, '70B', '#f97316')]

D_grid = np.logspace(9, 13, 200)   # 1B tokens up to 10T tokens

fig, ax = plt.subplots(figsize=(8.5, 5))
for N, label, color in model_sizes:
    C  = 6 * N * D_grid
    L  = chinchilla_loss(N, D_grid, A=A_fit, B=B_fit, E=E_fit)
    ax.loglog(C, L, label=f'{label} params', color=color, linewidth=2)

    # Optimal point: D* = 20N -> C* = 120 N^2.
    D_star = 20 * N
    C_star = 6 * N * D_star
    L_star = chinchilla_loss(N, D_star, A=A_fit, B=B_fit, E=E_fit)
    ax.scatter([C_star], [L_star], marker='*', s=160, color=color,
               edgecolor='#e2e8f0', linewidth=0.8, zorder=5)

ax.set_xlabel('Compute  C = 6 N D  (FLOPs)')
ax.set_ylabel('Loss  L')
ax.set_title('Chinchilla scaling: loss vs compute  (★ = compute-optimal per N)')
ax.legend()
plt.tight_layout(); plt.show()

## Key takeaways

- **Chinchilla loss law:** $L(N, D) = A/N^\alpha + B/D^\beta + E$ — each term
  captures one bottleneck (model capacity, data, irreducible noise).
- **Fitting** $(A, B, E)$ at known exponents is a *linear* least-squares problem
  — one `np.linalg.lstsq` recovers everything.
- **Compute-optimal recipe:** $D^* = 20 N^*$, giving $N^* = \sqrt{C / 120}$,
  $D^* = 20 N^*$ when $C = 6 N D$.
- Plotting $L$ vs $C$ on log-log axes makes power-law scaling visually obvious.

---
## ✏️ Your turn

### Exercise — implement `chinchilla_optimal(C)`

Implement a function that takes a training compute budget $C$ (FLOPs) and
returns the compute-optimal **(N\*, D\*)** under the Chinchilla rule
$D = 20 N$ with $C = 6 N D$.

$$C = 6 N D = 6 N (20 N) = 120 N^2 \;\Rightarrow\; N^* = \sqrt{C / 120}, \quad D^* = 20 N^*.$$

In [ ]:
import math

def chinchilla_optimal(C):
    """
    Return (N_opt, D_opt) for compute budget C (FLOPs) under the Chinchilla
    compute-optimal rule  D = 20 N,  C = 6 N D.

    Args:
        C : float, total training compute in FLOPs.

    Returns:
        (N_opt, D_opt) : floats. Parameter count and training-token count.
    """
    # TODO(you): derive  N* = sqrt(C / 120),  D* = 20 N*.
    ...

In [ ]:
# Test 1: C = 1.2e23 FLOPs  ->  N* should be roughly 3.16e10 (~32B), D* ~ 6.3e11 (~630B).
N_opt, D_opt = chinchilla_optimal(1.2e23)
assert math.isclose(N_opt, math.sqrt(1.2e23 / 120.0), rel_tol=1e-9), \
    f"N_opt mismatch: got {N_opt}"
assert math.isclose(D_opt, 20 * N_opt, rel_tol=1e-12), \
    f"D_opt must equal 20 * N_opt under the Chinchilla rule"

# Test 2: the C = 6 N D identity must hold.
C_check = 6 * N_opt * D_opt
assert math.isclose(C_check, 1.2e23, rel_tol=1e-9), \
    f"6 * N * D must equal C; got {C_check:.3e}"

# Test 3: doubling C scales N* by sqrt(2).
N_lo, _ = chinchilla_optimal(1e22)
N_hi, _ = chinchilla_optimal(2e22)
assert math.isclose(N_hi / N_lo, math.sqrt(2), rel_tol=1e-9), \
    f"Doubling C should multiply N* by sqrt(2); got ratio {N_hi/N_lo:.4f}"

print("✅ chinchilla_optimal is correct")
print(f"   At C = 1.2e23 FLOPs:  N* ≈ {N_opt/1e9:.1f}B params,  D* ≈ {D_opt/1e9:.0f}B tokens")

<details>
<summary>💡 Show solution</summary>

```python
def chinchilla_optimal(C):
    N_opt = math.sqrt(C / 120.0)
    D_opt = 20.0 * N_opt
    return N_opt, D_opt
```

Derivation: substitute $D = 20 N$ into $C = 6 N D$ to get $C = 120 N^2$,
then solve for $N$.

</details>